In [14]:
!pip3 install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.2 MB/s  0:00:005.9 MB/s eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [1]:
import json
def read_json(json_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
    return data

def write_json(json_file, data):
    with open(json_file, 'w+') as f:
        json.dump(data, f)

First, we will visualise the bounding boxes, landmarks, and segmentation the dataset provides for us below. The following code in this part is generated via Deepseek.

In [2]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

In [3]:
def visualize_clothing_data(image_path, json_data):
    """
    Visualize segmentation polygons and landmarks on the image
    """
    # Load the image
    image = Image.open(image_path)
    
    # Create figure and axes
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # Plot original image on first subplot
    ax1.imshow(image)
    ax1.set_title('Original Image')
    ax1.axis('off')
    
    # Plot annotated image on second subplot
    ax2.imshow(image)
    ax2.set_title('Annotated Image')
    
    # Colors for different items
    colors = {'item1': 'red', 'item2': 'blue'}
    
    for item_key, item_data in json_data.items():
        if item_key.startswith('item'):
            color = colors[item_key]
            category = item_data.get('category_name', 'Unknown')
            
            # Draw segmentation polygons
            for i, polygon in enumerate(item_data['segmentation']):
                # Convert flat list to pairs of coordinates
                points = np.array(polygon).reshape(-1, 2)
                
                # Draw polygon
                polygon_patch = patches.Polygon(
                    points, 
                    fill=True, 
                    alpha=0.3, 
                    facecolor=color,
                    edgecolor=color, 
                    linewidth=2,
                    label=f'{category} - Polygon {i+1}' if i == 0 else ""
                )
                ax2.add_patch(polygon_patch)
            
            # Draw bounding box
            bbox = item_data['bounding_box']
            x_min, y_min, x_max, y_max = bbox
            rect = patches.Rectangle(
                (x_min, y_min), 
                x_max - x_min, 
                y_max - y_min,
                linewidth=2, 
                edgecolor=color, 
                facecolor='none',
                linestyle='--',
                label=f'{category} - Bounding Box'
            )
            ax2.add_patch(rect)
            
            # Draw landmarks
            landmarks = item_data['landmarks']
            for i in range(0, len(landmarks), 3):
                x, y, visibility = landmarks[i:i+3]
                if visibility > 0:  # Only plot visible landmarks
                    ax2.plot(x, y, 'o', color=color, markersize=8, 
                            markeredgecolor='white', markeredgewidth=2)
                    ax2.annotate(str(i//3), (x, y), 
                               xytext=(5, 5), textcoords='offset points',
                               color='white', fontweight='bold',
                               bbox=dict(facecolor=color, alpha=0.7, pad=2))
            
            # Add text with item info
            info_text = f"{category}\nScale: {item_data['scale']}\nViewpoint: {item_data['viewpoint']}"
            ax2.text(x_min, y_min - 10, info_text, 
                    fontsize=8, color='white',
                    bbox=dict(facecolor=color, alpha=0.7, pad=2))
    
    # Add legend
    ax2.legend(loc='upper right', fontsize=8)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()

In [4]:
def visualize_segmentation_only(image_path, json_data):
    """
    Simplified visualization focusing only on segmentation polygons
    """
    # Load the image
    image = Image.open(image_path)
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    ax.imshow(image)
    
    # Colors for different polygons
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    # Plot all segmentation polygons
    for item_key, item_data in json_data.items():
        if item_key.startswith('item'):
            category = item_data.get('category_name', 'Unknown')
            
            print(f"\n{item_key} - {category}:")
            
            for i, polygon in enumerate(item_data['segmentation']):
                color = colors[i % len(colors)]
                
                # Convert flat list to pairs of coordinates
                points = np.array(polygon).reshape(-1, 2)
                
                print(f"  Polygon {i+1}: {len(points)} points")
                print(f"    Points: {points.tolist()}")
                
                # Draw polygon
                polygon_patch = patches.Polygon(
                    points, 
                    fill=True, 
                    alpha=0.3, 
                    facecolor=color,
                    edgecolor=color, 
                    linewidth=2,
                    label=f'{category} - Polygon {i+1}'
                )
                ax.add_patch(polygon_patch)
                
                # Plot points
                ax.plot(points[:, 0], points[:, 1], 'o', color=color, markersize=4)
    
    ax.set_title('Segmentation Visualization')
    ax.legend(loc='upper right', fontsize=8)
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [5]:
json_path = "deepfashiondataset/DeepFashion2/train/annos/000001.json"
json_data = read_json(json_path)
image_path = "deepfashiondataset/DeepFashion2/train/image/000001.jpg"

try:
    # Run both visualizations
    print("Creating full visualization with annotations...")
    visualize_clothing_data(image_path, json_data)
    
    print("\nCreating segmentation-only visualization...")
    visualize_segmentation_only(image_path, json_data)
    
except FileNotFoundError:
    print(f"Error: Could not find image at {image_path}")
    print("Please make sure the image file exists in the current directory")
except Exception as e:
    print(f"An error occurred: {e}")

FileNotFoundError: [Errno 2] No such file or directory: 'deepfashiondataset/DeepFashion2/train/annos/000001.json'

In [ ]:
# Bonus: If you want to save the visualization instead of showing it
def save_visualization(image_path, json_data, output_path="visualization.png"):
    """
    Save the visualization to a file
    """
    image = Image.open(image_path)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 12))
    ax.imshow(image)
    
    colors = ['red', 'blue']
    
    for idx, (item_key, item_data) in enumerate(json_data.items()):
        if item_key.startswith('item'):
            color = colors[idx % len(colors)]
            
            for polygon in item_data['segmentation']:
                points = np.array(polygon).reshape(-1, 2)
                polygon_patch = patches.Polygon(
                    points, fill=True, alpha=0.3, 
                    facecolor=color, edgecolor=color, linewidth=2
                )
                ax.add_patch(polygon_patch)
    
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Visualization saved to {output_path}")

The following will conduct "localisation", where we will localise where the fashion item is in the image. The following is not directly copied from GenAI, but GenAI is used for debugging and quick guidance.

In [28]:
!pip3 install ultralytics torch torchvision

In [3]:
# Initalising the paths
def training_set_imagespath(file_name):
    return f"deepfashiondataset/DeepFashion2/train/image/{file_name}"
def training_set_annospath(file_name):
    return f"deepfashiondataset/DeepFashion2/train/annos/{file_name}"

def validation_set_imagespath(file_name):
    return f"deepfashiondataset/DeepFashion2/validation/image/{file_name}"
def validation_set_annospath(file_name):
    return f"deepfashiondataset/DeepFashion2/validation/annos/{file_name}"

def testing_set_imagespath(file_name):
    return f"deepfashiondataset/DeepFashion2/test/image/{file_name}"
def testing_set_annospath(file_name):
    return f"deepfashiondataset/DeepFashion2/test/annos/{file_name}"


In [4]:
from ultralytics import YOLO
import os
from IPython.display import display, Image
from IPython import display
display.clear_output()


In [5]:
!yolo checks

Ultralytics 8.4.21 🚀 Python-3.14.3 torch-2.10.0 CPU (Apple M1)
Setup complete ✅ (8 CPUs, 16.0 GB RAM, 181.2/460.4 GB disk)

OS                     macOS-15.7.5-arm64-arm-64bit-Mach-O
Environment            Darwin
Python                 3.14.3
Install                pip
Path                   /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/ultralytics
RAM                    16.00 GB
Disk                   181.2/460.4 GB
CPU                    Apple M1
CPU count              8
GPU                    None
GPU count              None
CUDA                   None

numpy                  ✅ 2.4.3>=1.23.0
matplotlib             ✅ 3.10.8>=3.3.0
opencv-python          ✅ 4.13.0.92>=4.6.0
pillow                 ✅ 12.1.1>=7.1.2
pyyaml                 ✅ 6.0.3>=5.3.1
requests               ✅ 2.32.5>=2.23.0
scipy                  ✅ 1.17.1>=1.4.1
torch                  ✅ 2.10.0>=1.8.0
torch                  ✅ 2.10.0!=2.4.0,>=1.8.0; sys_platform == "win32"
torchvision     

<h1>We first are going to convert our raw .jpg images and .json anno files into YOLO-understandable format, i.e. in .yaml. </h1>

In [5]:
"""
We are going to move my files into the correct file structure
NEW FORMAT
dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
└── data.yaml
"""
import os
import shutil

def int_to_str(i):
    string_i = str(i)
    return "0"*(6-len(string_i))+string_i

In [ ]:
for i in range(137847,191961):
    filename = f'{int_to_str(i+1)}.jpg'
    original_filepath = f'deepfashiondataset/DeepFashion2/train/image/{filename}'
    new_filepath = f'deepfashiondataset/dataset_for_yolo/images/train/{filename}'
    shutil.move(original_filepath, new_filepath)

In [9]:
for i in range(191961):
    filename = f'{int_to_str(i+1)}.json'
    original_filepath = f'deepfashiondataset/DeepFashion2/train/annos/{filename}'
    new_filepath = f'deepfashiondataset/dataset_for_yolo/json/train/{filename}'
    shutil.move(original_filepath, new_filepath)
    if i+1 % int(191961/10) == 0:
        print("+10% done")

FileNotFoundError: [Errno 2] No such file or directory: 'deepfashiondataset/DeepFashion2/train/annos/000001.json'

In [44]:
for i in range(32153):
    filename = f'{int_to_str(i+1)}.jpg'
    original_filepath = f'deepfashiondataset/DeepFashion2/validation/image/{filename}'
    new_filepath = f'deepfashiondataset/dataset_for_yolo/images/val/{filename}'
    shutil.move(original_filepath, new_filepath)

In [11]:
for i in range(32153):
    filename = f'{int_to_str(i+1)}.json'
    original_filepath = f'deepfashiondataset/DeepFashion2/validation/annos/{filename}'
    new_filepath = f'deepfashiondataset/dataset_for_yolo/json/val/{filename}'
    shutil.move(original_filepath, new_filepath)
    if (i+1) % int(32153/10) == 0:
        print("+10% done")

FileNotFoundError: [Errno 2] No such file or directory: 'deepfashiondataset/DeepFashion2/validation/annos/000001.json'

<h1>All files are moved. We shall create our .yaml file.</h1>

In [7]:
yaml_filepath = f'deepfashiondataset/dataset_for_yolo/yolo_dataset.yaml'

In [9]:
from PIL import Image
import os.path

In [7]:
"""
Conversion of .json to .txt
.txt file format #<class_id> <x_center> <y_center> <width> <height>
"""
def convert_json_to_txt(json_pathfile, txt_pathfile, image_pathfile):
    json_data = read_json(json_pathfile)
    with open(txt_pathfile,'w+') as f:
        text_to_dump = ""
        # Loop through the respective items
        for key, value in json_data.items():
            if "item" in str(key): #It relates to a corresponding bounding box
                class_id = value["category_id"]-1
                bounding_box = value["bounding_box"]
                with Image.open(image_pathfile) as img:
                    max_width, max_height = img.size
                width = abs(bounding_box[0] - bounding_box[2])/max_width
                height = abs(bounding_box[1] - bounding_box[3])/max_height
                x_center = abs(bounding_box[0] + bounding_box[2])/(max_width*2)
                y_center = abs(bounding_box[1] + bounding_box[3])/(max_height*2)
                subtext_to_dump = f"{str(class_id)} {str(x_center)} {str(y_center)} {str(width)} {str(height)}"
                text_to_dump += subtext_to_dump + "\n"
        f.write(text_to_dump)
        # print(text_to_dump)

In [21]:
os.chdir('/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29')

In [12]:
# We do validation set
allitems = 1000 #32153
for i in range(allitems):
    filenameid = int_to_str(i+1)
    convert_json_to_txt(f'datasets/json/val/{filenameid}.json',
                        f'datasets/labels/val/{filenameid}.txt',
                        f'datasets/all_images/val/{filenameid}.jpg')
    if i % int(allitems/10) == 0:
        print(f'{str(int(100*i/allitems))}% done.')

0% done.
10% done.
20% done.
30% done.
40% done.
50% done.
60% done.
70% done.
80% done.
90% done.


In [13]:
# We do training set
allitems = 1000 #191961
for i in range(allitems):
    filenameid = int_to_str(i+1)
    convert_json_to_txt(f'datasets/json/train/{filenameid}.json',
                        f'datasets/labels/train/{filenameid}.txt',
                        f'datasets/all_images/train/{filenameid}.jpg')
    if i % int(allitems/10) == 0:
        print(f'{str(int(100*i/allitems))}% done.')

0% done.
10% done.
20% done.
30% done.
40% done.
50% done.
60% done.
70% done.
80% done.
90% done.


<h1>Now lets run YOLO</h1>

In [15]:
from ultralytics import YOLO

# Load a model
model = YOLO('yolov8n.pt')  # or yolov8s.pt, yolov8m.pt, etc.
full_path = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/datasets/data.yaml'
# Train the model
results = model.train(
    data=full_path,
    epochs=5,         # 50-150 epochs typical for large datasets
    imgsz=640,          # Standard for most applications
    batch=32           # Conservative for single GPU
)

New https://pypi.org/project/ultralytics/8.4.24 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.21 🚀 Python-3.14.3 torch-2.10.0 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/datasets/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, 

In [41]:
def IOU_accuracy(test_bounding_box, truth_bounding_box):
    x1,y1,x2,y2 = test_bounding_box
    X1,Y1,X2,Y2 = truth_bounding_box

    bx1, by1, bx2, by2 = min(x1,X1), min(y1,Y1), min(x2,X2), min(y2,Y2)
    intersection_area = abs(bx1-bx2)*abs(by1-by2)
    union_area = abs(x1-x2)*abs(y1-y2) + abs(X1-X2)*abs(Y1-Y2) - intersection_area
    return intersection_area/union_area


[[0, 0, 5, 5], [0, 0, 5, 4]]


In [65]:
import random

full_path = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/runs/detect/train9/weights/best.pt'
val_images_path = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/datasets/all_images/val/'
val_label_path = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/datasets/json/val/'
model = YOLO(full_path)
scores = {
    "scale1":{"sum_score":0, "count":0, "avg_score":0},
    "scale2":{"sum_score":0, "count":0, "avg_score":0},
    "scale3":{"sum_score":0, "count":0, "avg_score":0},
    "occlusion1":{"sum_score":0, "count":0, "avg_score":0},
    "occlusion2":{"sum_score":0, "count":0, "avg_score":0},
    "occlusion3":{"sum_score":0, "count":0, "avg_score":0},
    "zoom_in1":{"sum_score":0, "count":0, "avg_score":0},
    "zoom_in2":{"sum_score":0, "count":0, "avg_score":0},
    "zoom_in3":{"sum_score":0, "count":0, "avg_score":0},
    "viewpoint1":{"sum_score":0, "count":0, "avg_score":0},
    "viewpoint2":{"sum_score":0, "count":0, "avg_score":0},
    "viewpoint3":{"sum_score":0, "count":0, "avg_score":0},
    
    "sum_score":0, "count":0, "avg_score":0
    }
write_json('results.json', scores)
index = 0
range_size = 1000
for i in random.sample(range(1, 32153), range_size):
    index +=1
    results = model(f'{val_images_path}{int_to_str(i)}.jpg', verbose=False)
    json_data = read_json(f'{val_label_path}{int_to_str(i)}.json')
    
    #Process the truth bounding boxes
    json_boxes = []
    for key, value in json_data.items():
        if "item" in str(key): #It relates to a corresponding bounding box
            class_id = value["category_id"]-1
            scale = value["scale"]
            occlusion = value["occlusion"]
            zoom_in = value["zoom_in"]
            bounding_box = value["bounding_box"]
            viewpoint = value["viewpoint"]
            contents = {"scale":scale, "occlusion":occlusion, "zoom_in":zoom_in, "viewpoint":viewpoint, "bounding_box":[bounding_box[0], bounding_box[1], bounding_box[2], bounding_box[3]]}
            json_boxes.append(contents)
    
    #Process the testing bounding boxes
    test_boxes = []
    for result in results:
        if result.boxes is not None:
            boxes = result.boxes
            # print(f"Found {len(boxes)} objects")
            
            for box in boxes:
                # Get bounding box coordinates [x1, y1, x2, y2]
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                test_boxes.append([x1,y1,x2,y2])

    #Now compute the score
    for truth_box in json_boxes:
        test_boxes.sort(key=lambda x: IOU_accuracy(x,truth_box["bounding_box"]), reverse=True)
        if len(test_boxes) >0:
            score = IOU_accuracy(test_boxes[0],truth_box["bounding_box"])
            test_boxes.pop(0)
        else:
            score = 0
        scale, occlusion, zoom_in, viewpoint = json_boxes[0]["scale"], json_boxes[0]["occlusion"], json_boxes[0]["zoom_in"], json_boxes[0]["viewpoint"]
        scores["sum_score"], scores["count"] = scores["sum_score"]+score, scores["count"]+1
        scores["avg_score"] = scores["sum_score"]/scores["count"]
        if scale >= 1:
            dictkey = f"scale{str(scale)}"
            scores[dictkey]["sum_score"], scores[dictkey]["count"] = scores[dictkey]["sum_score"]+score, scores[dictkey]["count"]+1
            scores[dictkey]["avg_score"] = scores[dictkey]["sum_score"]/scores[dictkey]["count"]
        if occlusion >= 1:
            dictkey = f"occlusion{str(occlusion)}"
            scores[dictkey]["sum_score"], scores[dictkey]["count"] = scores[dictkey]["sum_score"]+score, scores[dictkey]["count"]+1
            scores[dictkey]["avg_score"] = scores[dictkey]["sum_score"]/scores[dictkey]["count"]
        if zoom_in >= 1:
            dictkey = f"zoom_in{str(zoom_in)}"
            scores[dictkey]["sum_score"], scores[dictkey]["count"] = scores[dictkey]["sum_score"]+score, scores[dictkey]["count"]+1
            scores[dictkey]["avg_score"] = scores[dictkey]["sum_score"]/scores[dictkey]["count"]
        if viewpoint >= 1:
            dictkey = f"viewpoint{str(viewpoint)}"
            scores[dictkey]["sum_score"], scores[dictkey]["count"] = scores[dictkey]["sum_score"]+score, scores[dictkey]["count"]+1
            scores[dictkey]["avg_score"] = scores[dictkey]["sum_score"]/scores[dictkey]["count"]
    
    
    if index%(range_size//10) == 0:
        print(index/range_size)
        write_json("results.json",scores)
print("DONE")

0.1
0.2
0.3
0.4
0.5
0.6
0.7
0.8
0.9
1.0
DONE


<h1>Cropping from YOLO localisation</h1>

Now that YOLO gives us the bounding boxes, we crop each detected garment out of the original image. These crops become the inputs for the second stage (ResNet multi-label attribute classification). Each crop is saved as `<image_id>_<det_idx>_<class_id>.jpg` so the class predicted by YOLO is preserved in the filename, and a JSON manifest records every crop's source image, box, class and confidence.

In [ ]:
"""
Crop garments out of an image using the trained YOLO detector.

For each detection we:
  1. Take the predicted xyxy box, clamp it to the image bounds with slight padding.
  2. Crop with PIL and save the patch.
  3. Record the source image, box, predicted class id and confidence
    so the downstream ResNet knows what each crop is.
"""
from PIL import Image
import os

def crop_detections(model, image_path, output_dir, pad=0.02, conf_thres=0.25):
    """Run YOLO on an image and save a cropped patch per detected box."""
    os.makedirs(output_dir, exist_ok=True)
    results = model(image_path, verbose=False, conf=conf_thres)

    image_id = os.path.splitext(os.path.basename(image_path))[0]
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        W, H = img.size
        crops_meta = []
        for r in results:
            if r.boxes is None:
                continue
            for det_idx, box in enumerate(r.boxes):
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cls_id = int(box.cls[0].item())
                conf = float(box.conf[0].item())

                # Pad then clamp to image bounds.
                bw, bh = x2 - x1, y2 - y1
                x1p = max(0, int(x1 - pad * bw))
                y1p = max(0, int(y1 - pad * bh))
                x2p = min(W, int(x2 + pad * bw))
                y2p = min(H, int(y2 + pad * bh))
                if x2p <= x1p or y2p <= y1p:
                    continue

                crop = img.crop((x1p, y1p, x2p, y2p))
                crop_name = f"{image_id}_{det_idx}_{cls_id}.jpg"
                crop.save(os.path.join(output_dir, crop_name))
                crops_meta.append({
                    "crop_file": crop_name,
                    "source_image": image_path,
                    "bbox_xyxy": [x1p, y1p, x2p, y2p],
                    "class_id": cls_id,
                    "confidence": conf,
                })
    return crops_meta

In [ ]:
# Batch-crop the validation set with the fine-tuned YOLO weights and write a manifest.
weights_path = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/runs/detect/train9/weights/best.pt'
val_images_dir = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/datasets/all_images/val/'
crops_out_dir = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/datasets/yolo_crops/val/'
manifest_path = '/Users/vas/Desktop/UniversityDocuments/COMP4471/project/comp4471project_group29/datasets/yolo_crops/val_manifest.json'

model = YOLO(weights_path)

os.makedirs(crops_out_dir, exist_ok=True)
manifest = []
n_images = 1000
for i in range(1, n_images + 1):
    image_path = f"{val_images_dir}{int_to_str(i)}.jpg"
    if not os.path.exists(image_path):
        continue
    manifest.extend(crop_detections(model, image_path, crops_out_dir))
    if i % (n_images // 10) == 0:
        print(f"{i}/{n_images} images cropped, {len(manifest)} crops so far")

write_json(manifest_path, manifest)
print(f"Saved {len(manifest)} crops to {crops_out_dir}")
print(f"Manifest written to {manifest_path}")